# 프로토콜 v1.2 — WS1 채점: 학습 곡선과 붓스트랩 비교 (탐색 분석, CPU)

CPU 노트북(PI-BDP 4가지, GB)과 GPU 노트북(TSMixer, TCN, S-pool)의 결과를 **같은 평가 엔진, 같은 순항 창, 같은 추출**로 맞춰 비교합니다.

- GPU 실행이 계산량 규칙으로 추출을 줄였으면(3 → 2, m = 3 제외), CPU 결과도 같은 추출만 남깁니다.
- 평가 엔진마다 추출 평균을 내고, 엔진들을 묶어 통합 RMSE를 계산합니다(v1.2 2.5).
- **검정:** m마다 PI-BDP V0와 GB·TSMixer·TCN의 통합 RMSE 차이를 계층 붓스트랩(서브셋 → 평가 엔진, 100,000회)으로 구합니다. 표현은 v1.1 9절 규칙을 따릅니다.
- **S-pool:** 같은 평가 엔진에서 V0(m = 2, 첫 조합)와 S-pool TSMixer를 같은 방식으로 비교합니다.
- **역전점:** m을 키워 가며 V0가 처음으로 "더 정확"이 아니게 되는 m을 비교 모델마다 보고합니다.
- m마다 평가 엔진 구성이 다르다는 점(m = 5는 고장 엔진이 많은 서브셋만)을 결과 옆에 적습니다.
- PI-BDP 변형(noWeibull, melding)과 사이클 단위 결과는 기술 통계로만 냅니다.
- **수리(v1.2 2.3 규칙의 사이클 단위 적용):** 표본 Weibull 적합이 극단값(형상 모수가 매우 큼)이면 어떤 사이클에서 모든 표본의 Weibull 로그 밀도가 −∞가 되어 재가중 가중치가 정의되지 않습니다(NaN). 이런 사이클은 "Weibull 실패 → noWeibull" 규칙대로 같은 표본을 재가중 없이 씁니다. 저장된 사이클 예측(raw)에서 해당 추출만 다시 채점하고, 다시 채점한 코드가 저장된 noWeibull·GB 점수를 그대로 재현하는지 먼저 확인합니다.
- **민감도:** 수리한 추출을 모든 모델에서 함께 빼고(짝 유지) 같은 검정을 다시 합니다.

**실행 전:** 이 노트북을 커밋한 주소를 `RUN_COMMIT_URL`에 넣으세요. 런타임은 CPU입니다.

In [ ]:
import os, sys, json, glob, time, hashlib, re
import numpy as np, pandas as pd
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/PI-BDP_NCMAPSS'
else:
    ROOT = os.environ.get('NCM_ROOT', './ncm')
W1 = f'{ROOT}/addendum_v12/ws1'; DL = f'{W1}/dl'; OUT = W1
URL = r'https://github\.com/.+/commit/[0-9a-f]{7,40}$'
ADDENDUM_V12_URL = 'https://github.com/goniworld/pibdp-ncmapss/commit/f01eecf82f5bc45ed9a90d44d8361b8dcb217d63'
RUN_COMMIT_URL = ''
assert re.match(URL, RUN_COMMIT_URL), '이 노트북을 커밋하고 주소를 넣으세요.'
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 22), b''): h.update(b)
    return h.hexdigest()
H1 = json.load(open(f'{W1}/sha256_ws1_cpu.json')); HD = json.load(open(f'{DL}/sha256_ws1_dl.json'))
assert sha(f'{W1}/ws1_draw_scores.csv') == H1['ws1_draw_scores.csv'] and sha(f'{DL}/ws1_dl_draw_scores.csv') == HD['files']['ws1_dl_draw_scores.csv'], '입력 파일이 실행 당시와 다름'
dec = json.load(open(f'{DL}/ws1_dl_compute_decision.json')); print('GPU 계획', dec['plan'], '| over_budget', dec['over_budget'])

In [ ]:
# === 수리: Weibull 재가중이 정의되지 않은 사이클 (v1.2 2.3 규칙을 사이클 단위로) ===
CPU = pd.read_csv(f'{W1}/ws1_draw_scores.csv', dtype={'m': str})
CAPS = dict(zip(['DS01-005', 'DS02-006', 'DS03-012', 'DS04', 'DS05', 'DS06', 'DS07', 'DS08a-009', 'DS08c-008'], [38, 34, 34, 43, 38, 41, 39, 34, 28]))
WIN = pd.read_csv(f'{W1}/ws1_dev_windows.csv')
WVAR = ['V0', 'meld0.5', 'meld1']; QCOLS = lambda v: [c for c in RAWCOLS if c.startswith(v + '_q')]
def rescore(g, s):
    # CPU 노트북 채점 셀과 같은 계산 (순항 창 sse, 사이클 sse)
    cap = CAPS[s]; g = g.sort_values('cycle'); eol = int((g.true + g.cycle).iloc[0])
    w = WIN[(WIN.subset == s) & (WIN.unit == int(g.heldout.iloc[0])) & (WIN.cycle <= eol)].sort_values('cycle')
    w = w[w.cycle > w.cycle.min()]; X = w.merge(g.assign(cycle=g.cycle + 1), on='cycle', how='inner'); assert len(X) == len(w)
    n = X.n_win.values.astype(float); y = np.minimum(eol - X.cycle.values, cap).astype(float); out = {}
    for md in ['V0', 'noWeibull', 'meld0.5', 'meld1', 'GB']:
        col = f'{md}_q500' if md != 'GB' else 'GB'
        p = np.clip(X[col].values - 1, 0, cap); out[f'{md}_sse'] = float(np.sum(n * (p - y) ** 2))
        out[f'{md}_cyc_sse'] = float(np.sum((g[col].values - g.true.values) ** 2))
    return out
bad = CPU[CPU[[f'{v}_sse' for v in WVAR]].isna().any(axis=1)]
print('재가중이 정의되지 않은 추출', len(bad)); print(bad[['subset', 'heldout', 'm', 'draw', 'note']].to_string(index=False))
REP = []
for s in sorted(bad.subset.unique()):
    Rw = pd.read_csv(f'{W1}/raw/{s}.csv.gz', dtype={'m': str}); RAWCOLS = list(Rw.columns)
    # 재구현 검사: 이 서브셋의 모든 추출에서 저장된 noWeibull·GB(그리고 NaN이 아닌 V0) 점수를 재현해야 함
    for (x, m, d), g in Rw.groupby(['heldout', 'm', 'draw']):
        r = rescore(g, s); o = CPU[(CPU.subset == s) & (CPU.heldout == x) & (CPU.m == m) & (CPU.draw == d)].iloc[0]
        for k in ['noWeibull_sse', 'GB_sse', 'noWeibull_cyc_sse', 'GB_cyc_sse'] + ([] if np.isnan(o.V0_sse) else ['V0_sse']):
            assert np.isclose(r[k], o[k], rtol=1e-9, atol=1e-6), f'재채점 불일치 {s} {x} {m} {d} {k}'
    for _, b in bad[bad.subset == s].iterrows():
        g = Rw[(Rw.heldout == b.heldout) & (Rw.m == b.m) & (Rw.draw == b.draw)].copy()
        for v in WVAR:
            nanc = g[QCOLS(v)].isna().any(axis=1)
            if nanc.any():
                g.loc[nanc, QCOLS(v)] = g.loc[nanc, [c.replace(v + '_', 'noWeibull_', 1) for c in QCOLS(v)]].values
            REP.append(dict(subset=s, heldout=b.heldout, m=b.m, draw=b.draw, variant=v, nan_cycles=int(nanc.sum()), n_cycles=len(g),
                            first_nan_cycle=int(g.cycle[nanc].min()) if nanc.any() else None, note=b.note))
        r = rescore(g, s); i = b.name
        for k, val in r.items(): CPU.loc[i, k] = val
REP = pd.DataFrame(REP); print(REP.to_string(index=False))
assert not CPU[[c for c in CPU.columns if c.endswith('_sse')]].isna().any().any(), '수리 뒤에도 NaN이 남음'
BADKEYS = bad[['subset', 'heldout', 'm', 'draw']].reset_index(drop=True)
CPU.to_csv(f'{W1}/ws1_draw_scores_repaired.csv', index=False); REP.to_csv(f'{W1}/ws1_repairs.csv', index=False)

In [ ]:
# === 같은 엔진·같은 추출로 맞추기 ===
DLD = pd.read_csv(f'{DL}/ws1_dl_draw_scores.csv', dtype={'m': str})   # CPU는 위 수리 셀에서 읽음
K = ['subset', 'heldout', 'm', 'draw']
T = DLD[DLD.scen == 'target']
keys = T[T.model == 'TSMixer'][K].drop_duplicates()
for md in ['TCN']:
    assert T[T.model == md][K].drop_duplicates().merge(keys).shape[0] == len(keys), f'{md}와 TSMixer의 추출 구성이 다름'
C1 = CPU.merge(keys, on=K)                                                # GPU 계획에 맞춘 CPU 추출
assert len(C1) == len(keys), 'CPU 결과에 없는 추출이 있음'
chk = C1[K + ['n']].merge(T[T.model == 'TSMixer'][K + ['n']], on=K, suffixes=('', '_dl')); assert np.allclose(chk.n, chk.n_dl), '창 수가 CPU와 GPU에서 다름'
MOD_CPU = ['V0', 'noWeibull', 'meld0.5', 'meld1', 'GB']
long = [C1[K + ['n', 'n_cyc']].assign(model=md, sse=C1[f'{md}_sse'], cyc_sse=C1[f'{md}_cyc_sse']) for md in MOD_CPU]
long += [T[T.model == md][K + ['n', 'n_cyc', 'sse', 'cyc_sse']].assign(model=md) for md in ['TSMixer', 'TCN']]
LG = pd.concat(long, ignore_index=True)
ENG = LG.groupby(['model', 'subset', 'heldout', 'm'])[['n', 'sse', 'n_cyc', 'cyc_sse']].mean().reset_index()   # 추출 평균
print('추출', len(keys), '| 평가 엔진 × m', ENG[ENG.model == 'V0'].shape[0])

In [ ]:
# === 학습 곡선 표 ===
M_ORDER = [m for m in ['2', '3', '5', 'all'] if m in set(ENG.m)]
MODELS = MOD_CPU + ['TSMixer', 'TCN']
def pooled(g, col='sse', n='n'): return float(np.sqrt(g[col].sum() / g[n].sum()))
CUR = pd.DataFrame([dict(m=m, model=md, rmse=pooled(g), cyc_rmse=pooled(g, 'cyc_sse', 'n_cyc'), n_engines=len(g))
                    for (m, md), g in ENG.groupby(['m', 'model'])])
TAB = CUR.pivot(index='m', columns='model', values='rmse').reindex(M_ORDER)[MODELS]; TAB['n_engines'] = CUR.groupby('m').n_engines.first()
TABC = CUR.pivot(index='m', columns='model', values='cyc_rmse').reindex(M_ORDER)[MODELS]
SUBT = pd.DataFrame([dict(subset=s, m=m, model=md, rmse=pooled(g)) for (s, m, md), g in ENG.groupby(['subset', 'm', 'model'])]).pivot_table(
    index=['subset', 'm'], columns='model', values='rmse')[MODELS]
TAB.to_csv(f'{OUT}/ws1_curve.csv'); TABC.to_csv(f'{OUT}/ws1_curve_cycle.csv'); SUBT.to_csv(f'{OUT}/ws1_curve_subset.csv')
print(TAB.round(3).to_string()); print(); print(SUBT.round(3).to_string())

In [ ]:
# === 계층 붓스트랩 (서브셋 → 평가 엔진), V0 - 모델, m마다 ===
def hier_boot(A, Bm, n_boot=100000, seed=1):
    U = A[['subset', 'heldout', 'sse', 'n']].merge(Bm[['subset', 'heldout', 'sse']], on=['subset', 'heldout'], suffixes=('_a', '_b'))
    subs = sorted(U.subset.unique()); rng = np.random.default_rng(seed)
    per = [U[U.subset == s][['sse_a', 'sse_b', 'n']].values for s in subs]
    SA = np.zeros(n_boot); SB = np.zeros(n_boot); N = np.zeros(n_boot)
    pick = rng.integers(0, len(subs), (n_boot, len(subs)))
    for j in range(len(subs)):
        for si, arr in enumerate(per):
            r = np.flatnonzero(pick[:, j] == si)
            if not len(r): continue
            tot = arr[rng.integers(0, len(arr), (len(r), len(arr)))].sum(1)
            SA[r] += tot[:, 0]; SB[r] += tot[:, 1]; N[r] += tot[:, 2]
    assert np.isfinite(U[['sse_a', 'sse_b', 'n']].values).all(), 'NaN 입력'
    d = np.sqrt(SA / N) - np.sqrt(SB / N); assert np.isfinite(d).all(); obs = float(np.sqrt(U.sse_a.sum() / U.n.sum()) - np.sqrt(U.sse_b.sum() / U.n.sum()))
    return dict(diff=obs, ci_lo=float(np.percentile(d, 2.5)), ci_hi=float(np.percentile(d, 97.5)),
                p_boot=float(min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))), n_engines=len(U), n_subsets=len(subs))
def wording(r):
    if r['ci_lo'] > 0: return 'PI-BDP less accurate'
    if r['ci_hi'] < 0: return 'PI-BDP more accurate (exploratory)'
    return 'comparable'
tests = []
for m in M_ORDER:
    A = ENG[(ENG.model == 'V0') & (ENG.m == m)]
    for md in ['GB', 'TSMixer', 'TCN']:
        r = hier_boot(A, ENG[(ENG.model == md) & (ENG.m == m)]); tests.append(dict(scenario='S-target', m=m, comparison=f'V0 - {md}', **r, wording=wording(r)))
# S-pool: 같은 평가 엔진, V0는 m = 2의 첫 조합
P = DLD[DLD.scen == 'pool']
if len(P):
    V2 = C1[(C1.m == '2') & (C1.draw == 0)][['subset', 'heldout', 'n', 'V0_sse']].rename(columns={'V0_sse': 'sse'})
    V2 = V2.merge(P[['subset', 'heldout']], on=['subset', 'heldout'])
    r = hier_boot(V2, P[['subset', 'heldout', 'sse', 'n']]); tests.append(dict(scenario='S-pool (m=2, draw 0)', m='2', comparison='V0 - TSMixer (pooled)', **r, wording=wording(r)))
    POOLT = pd.DataFrame([dict(model='V0 (m=2, draw 0)', rmse=pooled(V2)), dict(model='TSMixer S-pool', rmse=pooled(P)),
                          dict(model='TSMixer S-target (m=2, draw 0)', rmse=pooled(T[(T.model == 'TSMixer') & (T.m == '2') & (T.draw == 0)].merge(P[['subset', 'heldout']])))])
    POOLT.to_csv(f'{OUT}/ws1_pool.csv', index=False); print(POOLT.round(3).to_string(index=False))
TESTS = pd.DataFrame(tests); TESTS.to_csv(f'{OUT}/ws1_tests.csv', index=False); print(TESTS.round(4).to_string(index=False))
# 역전점: m을 키워 가며 V0가 처음으로 "더 정확"이 아니게 되는 m
REV = []
for md in ['GB', 'TSMixer', 'TCN']:
    t = TESTS[(TESTS.scenario == 'S-target') & (TESTS.comparison == f'V0 - {md}')].set_index('m').reindex(M_ORDER)
    first = next((m for m in M_ORDER if t.loc[m, 'wording'] != 'PI-BDP more accurate (exploratory)'), None)
    REV.append(dict(comparison=f'V0 - {md}', V0_more_accurate_at=[m for m in M_ORDER if t.loc[m, 'wording'].startswith('PI-BDP more')],
                    first_m_not_more_accurate=first, wording_by_m=dict(t.wording)))
REV = pd.DataFrame(REV); REV.to_csv(f'{OUT}/ws1_reversal.csv', index=False); print(REV.to_string(index=False))
# 민감도: 수리한 추출을 모든 모델에서 함께 뺌
LGX = LG.merge(BADKEYS.assign(_bad=1), on=K, how='left'); LGX = LGX[LGX._bad.isna()].drop(columns='_bad')
ENGX = LGX.groupby(['model', 'subset', 'heldout', 'm'])[['n', 'sse']].mean().reset_index()
sens = []
for m in M_ORDER:
    A = ENGX[(ENGX.model == 'V0') & (ENGX.m == m)]
    for md in ['GB', 'TSMixer', 'TCN']:
        r = hier_boot(A, ENGX[(ENGX.model == md) & (ENGX.m == m)]); sens.append(dict(scenario='S-target excl. repaired', m=m, comparison=f'V0 - {md}', **r, wording=wording(r)))
if len(P):
    bk = BADKEYS[(BADKEYS.m == '2') & (BADKEYS.draw == 0)][['subset', 'heldout']].assign(_bad=1)
    V2x = V2.merge(bk, how='left', on=['subset', 'heldout']); V2x = V2x[V2x._bad.isna()]
    r = hier_boot(V2x, P[['subset', 'heldout', 'sse', 'n']]); sens.append(dict(scenario='S-pool excl. repaired', m='2', comparison='V0 - TSMixer (pooled)', **r, wording=wording(r)))
SENS = pd.DataFrame(sens); SENS.to_csv(f'{OUT}/ws1_tests_excl_repaired.csv', index=False); print(SENS.round(4).to_string(index=False))

In [ ]:
# === 요약 + 해시 ===
txt = '\n'.join([f"Addendum v1.2 WS1 scoring (exploratory, dev data only) {time.strftime('%Y-%m-%d %H:%M:%S')}  addendum={ADDENDUM_V12_URL}  run={RUN_COMMIT_URL}",
    f"GPU plan {dec['plan']} (over budget: {dec['over_budget']}); draws used {len(keys)}; engines per m: {dict(TAB.n_engines)}",
    '', 'Learning curve, pooled window RMSE (eta=5% caps, t-1 rule for PI-BDP/GB; draw-averaged per held-out engine):', TAB.round(3).to_string(),
    '', 'Cycle-level uncapped RMSE (descriptive):', TABC.round(3).to_string(),
    '', 'Paired hierarchical bootstrap V0 - model, 100,000 resamples:', TESTS.round(4).to_string(index=False),
    '', 'Repaired cycles (Weibull weights undefined -> noWeibull for that cycle, v1.2 2.3 rule):', REP.to_string(index=False),
    '', 'Sensitivity: repaired draws removed from every model:', SENS.round(4).to_string(index=False),
    '', 'Reversal point:', REV.to_string(index=False),
    *(['', 'S-pool (same engines):', POOLT.round(3).to_string(index=False)] if len(P) else []),
    '', 'Per subset:', SUBT.round(3).to_string()])
open(f'{OUT}/ws1_summary.txt', 'w').write(txt)
H = {os.path.basename(p): sha(p) for p in [f'{OUT}/{n}' for n in ['ws1_curve.csv', 'ws1_curve_cycle.csv', 'ws1_curve_subset.csv', 'ws1_tests.csv', 'ws1_reversal.csv', 'ws1_summary.txt', 'ws1_repairs.csv', 'ws1_draw_scores_repaired.csv', 'ws1_tests_excl_repaired.csv'] + (['ws1_pool.csv'] if len(P) else [])]}
json.dump(H, open(f'{OUT}/sha256_ws1_score.json', 'w'), indent=1); print(txt)